# 01 — Inventory and splits (Day 1)

Thin runner: no logic here, only orchestration. All logic lives in
`sharp_har/` (inventory.py, windowing.py, splits.py).
Ref. `giorno1_inventory_splits_SPEC.md`.

## 1. Environment setup

In [ ]:
!pip install -q -r requirements.txt

import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml

set_seed(42)
print("git hash:", get_git_hash(REPO_DIR))

## 2. Mount Drive + staging

Data never enters the repo: it lives on Drive, gets copied and
unpacked locally. Training only reads from `/content`.

In [ ]:
import time
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
stage_dir.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for zip_name in paths_cfg["zips"]:
    src = drive_root / zip_name
    dst = Path("/content") / zip_name
    print(f"copying {src} -> {dst}")
    subprocess.run(["cp", str(src), str(dst)], check=True)
    with zipfile.ZipFile(dst) as zf:
        zf.extractall(stage_dir)
staging_seconds = time.time() - t0
print(f"staging completed in {staging_seconds:.1f}s (a datapoint for the day-2 gate)")

## 3. Naming reconnaissance

**Inspect the output before proceeding.** If the patterns don't
match `FILENAME_PATTERN` in `sharp_har/inventory.py`, fix the regex
there before running the next cell.

In [ ]:
from sharp_har.inventory import list_files, print_naming_patterns

files = list_files(stage_dir)
print(f"{len(files)} files found in {stage_dir}")
print_naming_patterns(files, n_examples=30)

## 4. Inventory

In [ ]:
from sharp_har.inventory import build_inventory

inventory_df = build_inventory(stage_dir, out_dir=REPO_DIR / "reports")

print("AR-set coverage:", sorted(inventory_df["ar_set"].unique()))
print("dtype distribution:")
print(inventory_df["dtype"].value_counts())
nan_pct = 100 * inventory_df["has_nan"].mean()
print(f"files with NaN: {nan_pct:.1f}%")

## 5. Day-1 checks / gate

Axes, AR-set coverage, activity×AR-set contingency, NaN policy
(stop if >5%), class decision, window-count sanity check.

In [ ]:
from sharp_har.inventory import (
    assert_axes, assert_coverage, build_contingency_table,
    apply_nan_policy, decide_classes,
)
from sharp_har.windowing import (
    count_windows, EXPECTED_WINDOWS_TRAIN_STRIDE, EXPECTED_WINDOWS_EVAL_STRIDE,
    TRAIN_STRIDE, EVAL_STRIDE,
)

assert_axes(inventory_df)

missing = assert_coverage(inventory_df)
assert not missing, f"missing AR-sets: {missing} — blocker, resolve before freezing the splits"

contingency = build_contingency_table(inventory_df, REPO_DIR / "reports" / "contingency.csv")
print(contingency)

inventory_clean = apply_nan_policy(inventory_df, out_dir=REPO_DIR / "reports")

classes_info = decide_classes(inventory_clean)
print(classes_info)

sample_n_frame = int(inventory_clean["n_frame"].median())
print("expected windows (train stride, median n_frame):",
      count_windows(sample_n_frame, stride=TRAIN_STRIDE), "expected ~", EXPECTED_WINDOWS_TRAIN_STRIDE)
print("expected windows (eval stride, median n_frame):",
      count_windows(sample_n_frame, stride=EVAL_STRIDE), "expected ~", EXPECTED_WINDOWS_EVAL_STRIDE)

## 6. P2-lab split (primary rotation, C1–C4)

Test = AR-7 (laboratory, S7 - the paper's hardest set). Val = 15% of train, stratified. Rare cells
pinned. Writes `splits/p2_lab.json` only if the per-cell
coverage assert passes.

In [ ]:
from sharp_har.splits import build_p2_rotation

p2_payload = build_p2_rotation(
    inventory_clean,
    out_path=REPO_DIR / "splits" / "p2_lab.json",
    labels=classes_info["labels"],
)
print("train:", len(p2_payload["train"]), "val:", len(p2_payload["val"]), "test:", len(p2_payload["test"]))
print("norm:", p2_payload["norm"])

## 7. P1-SHARP split (reproduction, C0)

Train = bedroom S1 (AR-1), all campaigns a/b/c. Val = 20% of S1. Test = the SHARP
generalization set. Writes `splits/p1_sharp.json`.

In [ ]:
from sharp_har.splits import build_p1_sharp

p1_payload = build_p1_sharp(
    inventory_clean,
    out_path=REPO_DIR / "splits" / "p1_sharp.json",
    c0_paper_set=classes_info["c0_paper_set"],
)
print("train:", len(p1_payload["train"]), "val:", len(p1_payload["val"]), "test:", len(p1_payload["test"]))
print("norm:", p1_payload["norm"])

## 8. Final commit

The artifacts produced in this run are **frozen** once
committed (§0.1 — never touched again):

- `splits/p2_lab.json`, `splits/p1_sharp.json`
- `reports/inventory.csv`, `reports/contingency.csv`,
  `reports/excluded_traces.csv`, `reports/name_to_arset.json`

```bash
cd sharp-har
git add splits/*.json reports/*.csv reports/*.json
git commit -m "day 1: inventory + frozen splits (p2_lab, p1_sharp)"
git push
```